In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn import metrics

In [2]:
np.random.seed(42)

r2_score_list = []
rmse_score_list = []
for i in range(10):
    data = pd.read_csv('data/data_DFT_mordredpca.csv')
    data['Alc_ID'] = data.index // 14
    shuffled_groups = data['Alc_ID'].unique()
    np.random.shuffle(shuffled_groups)
    train_groups = shuffled_groups[:10]
    test_groups = shuffled_groups[10:]
    train_data = data[data['Alc_ID'].isin(train_groups)].reset_index(drop=True)
    test_data = data[data['Alc_ID'].isin(test_groups)].reset_index(drop=True)

    y_train = pd.DataFrame(train_data['Yield'],columns=['Yield'])
    X_train = train_data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])
    y_test = pd.DataFrame(test_data['Yield'],columns=['Yield'])
    X_test = test_data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])
    
    a_X_train = (X_train - X_train.mean()) / X_train.std()
    a_X_test = (X_test - X_train.mean()) / X_train.std()
    a_X_train = a_X_train.dropna(how='any', axis=1)
    a_X_test = a_X_test[a_X_train.columns]
    param ={'hidden_layer_sizes':[(128,128,), (256,256,), (512,512,)], 'alpha':[0, 1, 2, 5]}
    reg = GridSearchCV(MLPRegressor(random_state=0, max_iter=1500, learning_rate_init=0.03),
                       param_grid=param, cv=5, n_jobs=12)
    reg.fit(a_X_train, y_train['Yield'])
    best = reg.best_estimator_
    #print(f'Run{i} model:', best)
    y_pred1 = best.predict(a_X_train)
    y_pred2 = best.predict(a_X_test)
    r2 = metrics.r2_score(y_test, y_pred2)
    rmse = metrics.root_mean_squared_error(y_test, y_pred2)
    print(f'Run{i} R2 (test):', r2, ', RMSE (test):', rmse)
    r2_score_list.append(r2)
    rmse_score_list.append(rmse)
print('==========(Result)==========')
print('Mean R2:', np.mean(r2_score_list))
print('SD R2:', np.std(r2_score_list))
print('Mean RMSE:', np.mean(rmse_score_list))
print('SD RMSE:', np.std(rmse_score_list))

Run0 R2 (test): 0.6625204220646767 , RMSE (test): 6.416505277497827
Run1 R2 (test): 0.2777446095944175 , RMSE (test): 23.53929965625379
Run2 R2 (test): 0.3828407956395684 , RMSE (test): 24.079784657696106
Run3 R2 (test): -0.4043541015531822 , RMSE (test): 31.290336183615956
Run4 R2 (test): 0.5957949674578862 , RMSE (test): 11.765180493643996
Run5 R2 (test): 0.2754672000508911 , RMSE (test): 22.86896958214295
Run6 R2 (test): -3.2878846730654683 , RMSE (test): 47.07068674613573
Run7 R2 (test): -6.912598129735016 , RMSE (test): 67.40658717370172
Run8 R2 (test): -0.27694569825220783 , RMSE (test): 18.719775013976424
Run9 R2 (test): -2.938282102071484 , RMSE (test): 22.586210011480603
==========(Result)==========
Mean R2: -1.1625696709869917
SD R2: 2.3461823303476055
Mean RMSE: 27.574333479614506
SD RMSE: 16.828625355893404
